# Function calling

Three ways to wire a tool to an LLM:

1. **Native function calling** — provider's first-class API (`tools=[...]`).
2. **JSON-schema-instructed prompting** — universal, but more brittle.
3. **Typed agent frameworks** — covered in `03-agentic-frameworks/`; this notebook shows the underlying mechanics so the frameworks aren't magic.

In [1]:
# %% Notebook bootstrap — never touches API keys directly.
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# CI / offline mode: replay cached responses instead of calling out.
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
import json
from shared.llm import Message, ToolSpec, complete

MODEL = 'openai/gpt-4o-mini'
NS = '00-foundations/02-function-calling'

## A tiny calculator tool
We define the schema once and pass it everywhere the model needs it. The schema is also what gets *hashed into the cache key*, so changing the parameter set is a cache-invalidating change.

In [3]:
calc = ToolSpec(
    name='calculator',
    description='Evaluate a simple arithmetic expression.',
    parameters={
        'type': 'object',
        'additionalProperties': False,
        'required': ['expression'],
        'properties': {'expression': {'type': 'string'}},
    },
)

def run_calculator(expression: str) -> str:
    # Whitelist: digits, ops, parens, spaces, dots only. Never eval untrusted input.
    allowed = set('0123456789+-*/(). ')
    assert set(expression) <= allowed, 'unsafe expression'
    return str(eval(expression))  # noqa: S307 - intentional for demo

## 1) Native function calling — single round trip

In [4]:
user_q = 'What is (2347 * 19) - 412?'
first = complete(
    model=MODEL, namespace=NS, tools=[calc],
    messages=[
        Message(role='system', content='You are a careful math helper. Use the calculator tool.'),
        Message(role='user', content=user_q),
    ],
)
print('tool_calls:', first.tool_calls)

tool_calls: [ToolCall(id='call_calc_1', name='calculator', arguments='{"expression": "(2347 * 19) - 412"}')]


The model picked a tool. We execute it, append the result, and call the model again so it can fold the result into a natural-language answer.

In [5]:
tc = first.tool_calls[0]
args = json.loads(tc.arguments)
result = run_calculator(args['expression'])
print('tool result:', result)

final = complete(
    model=MODEL, namespace=NS, tools=[calc],
    messages=[
        Message(role='system', content='You are a careful math helper. Use the calculator tool.'),
        Message(role='user', content=user_q),
        Message(role='assistant', content=f'TOOL_CALL: calculator({args["expression"]})'),
        Message(role='tool', name='calculator', content=result),
    ],
)
print(final.content)

tool result: 44181
(2347 * 19) - 412 evaluates to **44181**.


## 2) Multiple tools — the model picks one
Add a second tool. The model now has to choose.

In [6]:
from shared.loaders import load_corpus
CORPUS = load_corpus()

search = ToolSpec(
    name='search_papers',
    description='Search the synthetic arxiv corpus by keyword and return matching arxiv_ids.',
    parameters={
        'type': 'object',
        'additionalProperties': False,
        'required': ['query'],
        'properties': {
            'query': {'type': 'string'},
            'k': {'type': 'integer', 'default': 3},
        },
    },
)

def run_search(query: str, k: int = 3) -> list[str]:
    # Normalize hyphens so 'mixture of experts' matches 'Mixture-of-Experts'.
    q = query.lower().replace('-', ' ')
    hits = [d for d in CORPUS if q in (d.title + ' ' + d.abstract).lower().replace('-', ' ')]
    return [d.arxiv_id for d in hits[:k]]

In [7]:
user_q2 = 'Find papers in the corpus about mixture of experts and tell me how many.'
step1 = complete(
    model=MODEL, namespace=NS, tools=[search, calc],
    messages=[
        Message(role='system', content='You are a research assistant. Use the search_papers tool when needed.'),
        Message(role='user', content=user_q2),
    ],
)
print('chose tool:', step1.tool_calls[0].name)
ids = run_search(**json.loads(step1.tool_calls[0].arguments))
print('hits:', ids)

step2 = complete(
    model=MODEL, namespace=NS, tools=[search, calc],
    messages=[
        Message(role='system', content='You are a research assistant. Use the search_papers tool when needed.'),
        Message(role='user', content=user_q2),
        Message(role='assistant', content='TOOL_CALL: search_papers(mixture of experts)'),
        Message(role='tool', name='search_papers', content=json.dumps(ids)),
    ],
)
print(step2.content)

chose tool: search_papers
hits: ['synth-001']
I found **1** paper in the corpus about mixture-of-experts: `synth-001` (Routing-Aware Sparse Mixture-of-Experts for Latency-Bounded Inference).


## 3) JSON-schema-instructed fallback
When the provider has no native tool API, you can describe tools in the system prompt and force JSON output. It works, but the parsing is fragile and you lose the structured `tool_calls` field — you have to demux yourself.

In [8]:
instructed = (
    'You have ONE tool available:\n'
    '  calculator(expression: str) -> number\n\n'
    'If you need it, reply with ONLY a JSON object {"tool": "calculator", "args": {"expression": "..."}}.\n'
    'Otherwise reply with {"answer": "..."}.'
)
raw = complete(
    model=MODEL, namespace=NS,
    messages=[
        Message(role='system', content=instructed),
        Message(role='user', content=user_q),
    ],
    response_format={'type': 'json_object'},
).content
payload = json.loads(raw)
print(payload)
if payload.get('tool') == 'calculator':
    print('result:', run_calculator(payload['args']['expression']))

{'tool': 'calculator', 'args': {'expression': '(2347 * 19) - 412'}}
result: 44181


## Takeaways
* Use **native function calling** whenever the provider supports it — you get structured `tool_calls`, parallel tool selection, and (often) constrained decoding.
* Fall back to **JSON-schema-instructed prompting + `response_format=json_object`** only when you have to. Build a thin parser around it.
* Tool *schemas* are part of your prompt and should be hashed into your cache key — the `shared.llm` shim does this automatically.